# Práctico 4 – RAG básico y sus fallas

**Magíster en Inteligencia Artificial UC · INF3880 – Inteligencia Artificial Agéntica**

**Profesores:** Vicente Domínguez, Antonio Ossa · **Ayudante:** Michelle Madrid

**Nombre del alumno:** ESCRIBIR NOMBRE AQUÍ

> Este práctico **no es evaluado**. Son ejercicios propuestos para trabajar en el Módulo 2 de la clase 4 (lunes 09/11/2026) y terminar en casa si hace falta. La versión con soluciones se publica después de la clase.

Construiremos un pipeline **RAG** sobre el manual operativo de *Andina Cargo*: embeddings de Gemini, chunking, un índice vectorial mínimo en numpy, recuperación y respuesta con citas. Mediremos **precision@k** y provocaremos a propósito las fallas típicas de RAG. Seguimos con la **API gratuita de Gemini** y tu misma key.


## Configuración inicial

Usaremos la **API de Gemini en su nivel gratuito** (no requiere tarjeta de crédito). Cada alumno crea su propia key; toma unos 2 minutos.

### Paso A · Crear tu API key en Google AI Studio
1. Abre **https://aistudio.google.com/apikey** e inicia sesión con tu cuenta de Google (personal; la cuenta institucional puede tener AI Studio bloqueado).
2. La primera vez te pedirá **aceptar los términos de servicio** de la Gemini API. Al aceptarlos, Google crea automáticamente un proyecto por defecto.
3. Haz clic en **"Create API key"** (si ya tienes un proyecto, elige "Create API key in existing project" o el proyecto por defecto).
4. Copia la key con el botón **"Copiar clave"** (las keys actuales empiezan con `AQ.`; las antiguas con `AIza`). Guárdala en un lugar seguro: la vas a necesitar en el paso B. Si la pierdes, puedes crear otra desde la misma página.
5. Opcional pero recomendado: en la lista de keys, pasa el mouse sobre la etiqueta **"Unrestricted"** → **"Add restrictions"** → **"Restrict to Gemini API only"** → **"Restrict key"**.

> ¿Dónde ver los límites y el uso? En AI Studio, menú lateral → **Dashboard** → **Usage & Billing** (o https://aistudio.google.com/usage). Ahí aparece el plan "Free" y las cuotas por modelo.
>
> ¿Problemas? Si el botón "Create API key" no aparece, suele ser porque la cuenta es de una organización (Workspace) que restringe AI Studio: usa una cuenta personal de Gmail. Documentación oficial: https://ai.google.dev/gemini-api/docs/api-key

### Paso B · Guardar la key en Secrets de Colab
1. En este notebook, haz clic en el ícono de la **llave 🔑** de la barra lateral izquierda ("Secrets").
2. **"+ Add new secret"**: en *Name* escribe exactamente `GEMINI_API_KEY`; en *Value* pega la key.
3. Activa el interruptor **"Notebook access"** para este notebook.
4. **Nunca pegues la key en una celda, ni la subas a un repositorio, ni la compartas.** Trátala como una contraseña: si se filtra, crea una nueva y borra la antigua en AI Studio.

### Paso C · Instalar y configurar
Instalamos la librería `google-genai` y definimos el modelo en una sola variable `MODEL` para poder cambiarlo sin tocar el resto del notebook.

> ⚠️ **Dos cosas que debes saber del nivel gratuito.** (1) Según los términos de Google, los prompts y respuestas del nivel sin pago **pueden usarse para mejorar sus productos y ser revisados por personas**: no envíes datos personales, confidenciales ni de tu empresa. (2) Tiene **límites de uso por minuto y por día**; si recibes un error 429 (*resource exhausted*), espera un momento y reintenta. Los límites de tu proyecto se ven en AI Studio.

In [ ]:
%%capture
!pip install -q --upgrade google-genai pandas numpy

In [ ]:
import os, time
from google.colab import userdata

# 1) Leer la key desde Secrets, con mensajes claros si algo falta
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except userdata.SecretNotFoundError:
    raise SystemExit("No existe un secreto llamado GEMINI_API_KEY. Créalo en el panel de Secrets (ícono de la llave) con ese nombre exacto.")
except userdata.NotebookAccessError:
    raise SystemExit("El secreto GEMINI_API_KEY existe pero no tiene 'Notebook access' activado para este notebook.")
assert GEMINI_API_KEY and len(GEMINI_API_KEY) > 20, \
    "El secreto GEMINI_API_KEY está vacío. Pega la key completa desde https://aistudio.google.com/apikey (botón 'Copiar clave')."
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# 2) Elegir el modelo: buscamos uno que esté disponible en TU proyecto gratuito y que respete `temperature`
#    (los modelos Gemini 3.6+ y 4 la ignoran, por eso preferimos Flash-Lite 3.5 o 2.5)
from google import genai
client = genai.Client()
PREFERIDOS = ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.8-flash"]
disponibles = []
for m in client.models.list():
    acciones = getattr(m, "supported_actions", None) or []
    if not acciones or "generateContent" in acciones:
        disponibles.append(m.name.replace("models/", ""))
MODEL = next((p for p in PREFERIDOS if p in disponibles), None)
if MODEL is None:
    print("Ninguno de los modelos preferidos está disponible. Modelos de tu proyecto:", disponibles[:30])
    MODEL = next((d for d in disponibles if "flash" in d), disponibles[0])
RESPALDO = [p for p in PREFERIDOS if p in disponibles and p != MODEL]   # por si el modelo elegido falla (error 500/503)
print("Modelo elegido:", MODEL, "| respaldo:", RESPALDO)

# Precios de referencia del nivel DE PAGO, en USD por 1M tokens (https://ai.google.dev/gemini-api/docs/pricing).
# En el nivel gratuito no pagas, pero calcularemos cuánto costaría para entender la economía de un LLM.
PRECIOS = {
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
    "gemini-2.5-flash":      {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}

def costo_usd(input_tokens, output_tokens, model=None):
    p = PRECIOS.get(model or MODEL, PRECIOS["gemini-3.5-flash-lite"])   # si el modelo no está en la tabla, usa un precio de referencia
    return ((input_tokens or 0) * p["input"] + (output_tokens or 0) * p["output"]) / 1_000_000

print("Listo. Modelo:", MODEL)

Verificamos que la key funciona:

In [ ]:
from google.genai import types

def generar(contents, reintentos=3, **config):
    """Llama al modelo con manejo de los errores típicos del nivel gratuito:
    - 429 (límite de uso): espera y reintenta.
    - 500/503 (servidor saturado o modelo no disponible en ese momento): reintenta con esperas crecientes
      y, si sigue fallando, cambia al siguiente modelo de RESPALDO."""
    global MODEL
    for intento in range(reintentos + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents,
                                                  config=types.GenerateContentConfig(**config))
        except Exception as e:
            msg = str(e)
            if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) and intento < reintentos:
                print("  (límite de uso alcanzado, esperando 20 s...)"); time.sleep(20)
            elif any(k in msg for k in ("500", "503", "INTERNAL", "UNAVAILABLE", "overloaded", "not available")):
                if intento < reintentos:
                    espera = 5 * 2 ** intento          # 5, 10, 20 s
                    print(f"  (error transitorio del servidor, reintentando en {espera} s...)"); time.sleep(espera)
                elif RESPALDO:
                    anterior, MODEL = MODEL, RESPALDO.pop(0)
                    print(f"  ⚠️ {anterior} sigue fallando; cambiamos a {MODEL} y seguimos.")
                    return generar(contents, reintentos=reintentos, **config)
                else:
                    raise SystemExit("Gemini devolvió errores 500/503 en todos los modelos disponibles. Suele ser saturación temporal: espera unos minutos y vuelve a ejecutar esta celda.")
            elif "404" in msg or "NOT_FOUND" in msg:
                raise SystemExit(f"El modelo {MODEL} no está disponible en tu proyecto. Cambia MODEL por otro de la lista de la celda anterior.")
            elif "403" in msg or "401" in msg or "API_KEY_INVALID" in msg:
                raise SystemExit("La API key fue rechazada. Verifica que la copiaste completa desde https://aistudio.google.com/apikey y que no está restringida a otra API.")
            else:
                raise

print(generar("Responde solo: OK").text)

## Parte guiada 1 · Los documentos

Seguimos con **Andina Cargo**. Esta vez el asistente debe responder preguntas sobre el **manual operativo** de la empresa: despachos, devoluciones, reclamos, tarifas, bodega, seguridad y facturación. Son 8 documentos cortos en Markdown; uno es una **versión antigua de las tarifas** que contradice a la vigente. Lo dejamos a propósito: vamos a ver cómo falla RAG con él.

In [ ]:
DOCS = {
"despachos.md": """# Política de despachos
## Cobertura y tiempos
Los tiempos de despacho en días hábiles desde la salida de bodega son: Santiago 1, Valparaíso 2, Temuco 3, Antofagasta 4 y Punta Arenas 7. Zonas sin cobertura se cotizan caso a caso.
## Hora de corte
Los pedidos confirmados antes de las 15:00 salen el mismo día hábil. Los confirmados después del corte se procesan el día hábil siguiente.
## Seguimiento
Cada pedido tiene un identificador con formato P-1234. El estado se consulta en el portal de clientes o a través del asistente interno.""",

"devoluciones.md": """# Política de devoluciones y cambios
## Plazo
El plazo es de 10 días hábiles contados desde la entrega. El producto debe estar sin uso y con su embalaje original.
## Quién paga el flete
Si la devolución se debe a un error de despacho o a un producto dañado, Andina Cargo paga el flete de retorno. Si es por cambio de opinión del cliente, el flete lo paga el cliente.
## Reembolso
El reembolso se emite dentro de 5 días hábiles desde la recepción del producto en bodega, mediante nota de crédito o devolución al medio de pago.""",

"reclamos.md": """# Procedimiento de reclamos
## Canal
Los reclamos se ingresan por el portal de clientes o al correo reclamos@andinacargo.example, indicando el id del pedido.
## Plazos de respuesta
La primera respuesta se entrega dentro de 24 horas hábiles y la resolución dentro de 5 días hábiles.
## Daño en tránsito
Se debe adjuntar fotografía del embalaje y del producto junto con la guía de despacho. La compensación se realiza mediante nota de crédito por el valor declarado.""",

"tarifas.md": """# Tarifas de despacho 2026
## Zona central
Hasta 5 kg: $4.990. De 5 a 20 kg: $8.990. De 20 a 50 kg: $14.990. Sobre 50 kg se cotiza por pallet.
## Recargos
Zona extremo norte (Antofagasta y más al norte) y extremo sur (Punta Arenas): recargo del 40 % sobre la tarifa de zona central.
## Despacho gratuito
Pedidos sobre $150.000 tienen despacho gratuito en zona central.""",

"bodega.md": """# Operación de bodega
## Horario
La recepción de mercadería es de lunes a viernes entre 8:00 y 17:00. Fuera de ese horario se agenda para el día siguiente.
## Reserva de stock
El stock se reserva al confirmar el pedido. Si al preparar el pedido se detecta un quiebre de stock, se avisa al cliente dentro de 24 horas y se le ofrece despacho parcial o espera.
## Inventario
Se realiza inventario cíclico semanal por familia de productos; las diferencias se ajustan el mismo día.""",

"seguridad.md": """# Seguridad y mercancías restringidas
## Prohibidas
No se transportan baterías de litio sueltas, aerosoles, líquidos inflamables ni productos químicos sin ficha de seguridad.
## Frágiles
Deben rotularse con la etiqueta de frágil y embalarse con doble protección. Andina Cargo no se hace responsable por frágiles sin rotular.
## Seguro
Todo envío incluye seguro hasta $500.000. Para valores superiores se contrata seguro adicional equivalente al 1 % del valor declarado.""",

"facturacion.md": """# Facturación y pagos
## Emisión
La factura electrónica se emite al momento del despacho y se envía al correo registrado del cliente.
## Plazo de pago
Los clientes con línea de crédito aprobada pagan a 30 días. Los demás pagan al confirmar el pedido.
## Notas de crédito
Las devoluciones y compensaciones se documentan con nota de crédito dentro de 5 días hábiles.""",

"tarifas_2025_antigua.md": """# Tarifas de despacho 2025 (versión OBSOLETA)
## Zona central
Hasta 5 kg: $3.990. De 5 a 20 kg: $7.490. De 20 a 50 kg: $12.990.
## Despacho gratuito
Pedidos sobre $120.000 tienen despacho gratuito en zona central.""",
}
import pathlib
pathlib.Path("docs").mkdir(exist_ok=True)
for nombre, texto in DOCS.items():
    pathlib.Path("docs", nombre).write_text(texto, encoding="utf-8")
print(len(DOCS), "documentos en docs/ (nota: uno es una versión antigua y contradictoria, a propósito)")

## Parte guiada 2 · Embeddings y similitud

Un **embedding** es un vector de números que representa el *significado* de un texto: textos parecidos quedan cerca en ese espacio, aunque no compartan palabras. La cercanía se mide con **similitud coseno** (1 = misma dirección, 0 = nada que ver).

Gemini tiene modelos de embeddings separados de los de texto. Pedimos vectores de 768 dimensiones y los normalizamos, así la similitud coseno es simplemente el producto punto.

In [ ]:
import numpy as np
from google.genai import types

PAUSA = 1.5   # segundos entre llamadas al modelo de texto (límite por minuto del nivel gratuito)

# Elegimos un modelo de embeddings disponible en tu proyecto (distinto al modelo de texto)
EMB_PREFERIDOS = ["gemini-embedding-001", "gemini-embedding-2"]
emb_disponibles = [m.name.replace("models/", "") for m in client.models.list()
                   if "embedContent" in (getattr(m, "supported_actions", None) or [])]
EMB_MODEL = next((p for p in EMB_PREFERIDOS if p in emb_disponibles), None) or (emb_disponibles[0] if emb_disponibles else None)
assert EMB_MODEL, f"Tu proyecto no tiene modelos de embeddings disponibles. Modelos: {emb_disponibles}"
print("Modelo de embeddings:", EMB_MODEL, "| disponibles:", emb_disponibles)
DIM = 768   # los modelos Gemini permiten elegir la dimensión (128 a 3072); 768 basta para este práctico

def embed(textos, tipo="RETRIEVAL_DOCUMENT", reintentos=3):
    """Convierte una lista de textos en una matriz (n, DIM) de vectores normalizados.
    `tipo` le dice al modelo si son documentos a indexar o una consulta: mejora la búsqueda."""
    if EMB_MODEL.startswith("gemini-embedding-2"):   # este modelo no acepta task_type y agrupa partes: una Content por texto
        prefijo = "task: search result | query: " if tipo == "RETRIEVAL_QUERY" else "title: none | text: "
        contents = [types.Content(parts=[types.Part(text=prefijo + t)]) for t in textos]
        config = types.EmbedContentConfig(output_dimensionality=DIM)
    else:
        contents, config = textos, types.EmbedContentConfig(task_type=tipo, output_dimensionality=DIM)
    for intento in range(reintentos + 1):
        try:
            r = client.models.embed_content(model=EMB_MODEL, contents=contents, config=config)
            E = np.array([e.values for e in r.embeddings], dtype=float)
            return E / np.linalg.norm(E, axis=1, keepdims=True)   # normalizados: así similitud coseno = producto punto
        except Exception as e:
            if any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "500", "503", "UNAVAILABLE")) and intento < reintentos:
                print("  (límite de uso o error transitorio, esperando 15 s...)"); time.sleep(15)
            else:
                raise

frases = ["¿Cuánto demora un envío a Temuco?", "tiempo de entrega al sur de Chile", "El notebook se cierra solo", "plazo para devolver un producto"]
E = embed(frases, "SEMANTIC_SIMILARITY")
print("forma de la matriz de embeddings:", E.shape)

def coseno(a, b): return float(a @ b)     # vectores ya normalizados
for i in range(1, len(frases)):
    print(f"sim('{frases[0]}', '{frases[i]}') = {coseno(E[0], E[i]):.3f}")

💡 **Observa:** "tiempo de entrega al sur de Chile" queda más cerca de la pregunta por Temuco que "el notebook se cierra solo", aunque no comparten ninguna palabra. Eso permite **buscar por significado** en vez de por palabras exactas. La frase sobre devoluciones queda en un punto intermedio: habla de plazos, pero de otro proceso.

## Parte guiada 3 · Pipeline RAG: chunking → índice → recuperación → respuesta con citas

**RAG** (*retrieval-augmented generation*) es: trocear los documentos (*chunking*), convertir cada trozo en embedding, guardarlos en un **índice**, y al llegar una pregunta recuperar los k trozos más parecidos para que el modelo responda **solo con ellos**, citando la fuente.

Implementamos el índice en 20 líneas con numpy: un *vector store* no es más que esto (más escala, persistencia y filtros). Empezamos con el chunking más ingenuo: trozos de tamaño fijo.

In [ ]:
class Indice:
    """Un vector store mínimo: una matriz de embeddings + los textos y metadatos de cada chunk.
    Chroma, pgvector o Pinecone hacen exactamente esto, pero a escala de millones de vectores."""
    def __init__(self):
        self.ids, self.textos, self.metas, self.E = [], [], [], None

    def agregar(self, ids, textos, metas, lote=50):
        for i in range(0, len(textos), lote):               # embebemos de a 50 textos por llamada
            nuevos = embed(textos[i:i+lote], "RETRIEVAL_DOCUMENT")
            self.E = nuevos if self.E is None else np.vstack([self.E, nuevos])
        self.ids += ids; self.textos += textos; self.metas += metas

    def buscar(self, pregunta, k=3, filtro=None):
        """Devuelve los k chunks más similares a la pregunta (opcionalmente solo los que pasan `filtro(meta)`)."""
        q = embed([pregunta], "RETRIEVAL_QUERY")[0]
        sims = self.E @ q
        out = []
        for i in np.argsort(-sims):
            if filtro and not filtro(self.metas[i]): continue
            out.append({"id": self.ids[i], "doc": self.metas[i]["doc"], "texto": self.textos[i], "sim": round(float(sims[i]), 3)})
            if len(out) == k: break
        return out

def chunk_fijo(texto, tam=250, solape=40):
    """El chunking más ingenuo: trozos de tamaño fijo con un poco de solapamiento."""
    chunks, i = [], 0
    while i < len(texto):
        chunks.append(texto[i:i+tam]); i += tam - solape
    return chunks

def indexar(fn_chunk, meta_extra=None):
    """Trocea cada documento con fn_chunk y lo agrega al índice con su metadata."""
    idx = Indice()
    ids, textos, metas = [], [], []
    for doc, texto in DOCS.items():
        for j, ch in enumerate(fn_chunk(texto)):
            ids.append(f"{doc}#{j}"); textos.append(ch)
            metas.append({"doc": doc, **(meta_extra(doc) if meta_extra else {})})
    idx.agregar(ids, textos, metas)
    print(f"{len(ids)} chunks indexados con {fn_chunk.__name__}")
    return idx

idx_fijo = indexar(chunk_fijo)
for c in idx_fijo.buscar("¿Cuánto cuesta despachar un paquete de 3 kilos?"):
    print(f"[{c['id']}] sim={c['sim']}  {c['texto'][:80]!r}")

💡 **Observa** los fragmentos recuperados para la pregunta de los 3 kilos: probablemente aparecen las tarifas 2026 **y** las 2025 obsoletas, y algún trozo cortado a mitad de frase. Los embeddings no saben qué versión está vigente.

In [ ]:
SYSTEM_RAG = """Eres el asistente interno de Andina Cargo. Respondes SOLO con la información del contexto entregado.
Cita el id del fragmento que usas entre corchetes, por ejemplo [tarifas.md#1].
Si el contexto no alcanza para responder, di explícitamente que no está en los documentos. No inventes montos ni plazos."""

def responder_rag(idx, pregunta, k=3, verbose=False, filtro=None):
    """Recupera k fragmentos y pide al modelo una respuesta basada solo en ellos (con citas)."""
    ctx = idx.buscar(pregunta, k, filtro)
    if verbose:
        for c in ctx: print(f"  [{c['id']}] sim={c['sim']}  {c['texto'][:90]!r}")
    contexto = "\\n\\n".join(f"[{c['id']}]\\n{c['texto']}" for c in ctx)
    r = generar(f"Contexto:\\n{contexto}\\n\\nPregunta: {pregunta}", system_instruction=SYSTEM_RAG, temperature=0)
    time.sleep(PAUSA)
    return r.text, ctx

resp, ctx = responder_rag(idx_fijo, "¿Desde qué monto el despacho es gratis?", verbose=True)
print("\\n", resp)

💡 **Observa:** el modelo responde solo con el contexto y cita el fragmento. Si le llegaron las dos versiones de tarifas, fíjate si eligió la correcta, si mezcló ambas o si avisó de la contradicción. Ninguna de las tres cosas depende de "la calidad del modelo": depende de qué le recuperamos.

## Parte guiada 4 · Medir la recuperación: precision@k

No evaluamos RAG "a ojo". Para 10 preguntas sabemos en qué documento está la respuesta. **precision@k** = fracción de los k fragmentos recuperados que pertenecen al documento correcto. Es la métrica más simple de recuperación; en la clase vimos también recall@k y fidelidad de la respuesta.

In [ ]:
import pandas as pd

EVAL = [
 ("¿Cuántos días hábiles tarda un despacho a Temuco?", "despachos.md"),
 ("¿Hasta qué hora puedo confirmar un pedido para que salga hoy?", "despachos.md"),
 ("¿Cuánto plazo tengo para devolver un producto?", "devoluciones.md"),
 ("¿Quién paga el flete si me arrepiento de la compra?", "devoluciones.md"),
 ("¿En cuánto tiempo responden un reclamo?", "reclamos.md"),
 ("¿Cuánto cuesta despachar un paquete de 3 kilos?", "tarifas.md"),
 ("¿Desde qué monto el despacho es gratis?", "tarifas.md"),
 ("¿Qué pasa si no hay stock de lo que pedí?", "bodega.md"),
 ("¿Puedo enviar baterías de litio?", "seguridad.md"),
 ("¿Cuál es el plazo de pago de las facturas?", "facturacion.md"),
]

def precision_at_k(idx, k=3, filtro=None):
    filas = []
    for preg, doc_ok in EVAL:
        ctx = idx.buscar(preg, k, filtro)
        filas.append({"pregunta": preg, "doc_correcto": doc_ok, "recuperados": [c["doc"] for c in ctx],
                      "precision@k": sum(c["doc"] == doc_ok for c in ctx) / k})
    df = pd.DataFrame(filas)
    print(f"precision@{k} promedio: {df['precision@k'].mean():.2f}")
    return df

df_fijo = precision_at_k(idx_fijo)
df_fijo

💡 **Observa:** las preguntas de tarifas recuperan también la versión **obsoleta** (baja la precisión aunque "la respuesta esté"), y varios chunks de tamaño fijo cortaron una sección a la mitad y perdieron su título ("## Plazo" sin saber de qué política es). Son dos de las limitaciones de RAG de la clase: **pérdida de contexto** y **documentos contradictorios**. En los ejercicios las vamos a atacar.

## Ejercicios propuestos

De dificultad creciente. El último siempre es abierto. Si te trabas, revisa las pistas y la parte guiada; la solución se publica después de la clase.

### Ejercicio 1 – Chunking por secciones vs. tamaño fijo

Implementa `chunk_secciones(texto)` que divida cada documento por sus encabezados `## ` y anteponga a cada sección el título del documento (la línea `# ...`) para no perder contexto. Indexa con `indexar(chunk_secciones)` y compara precision@3 con la de tamaño fijo, pregunta por pregunta. ¿Cuáles mejoraron y por qué?

**Pistas:** `texto.split('\n## ')` deja el título en la posición 0 y las secciones después. Para comparar, arma un DataFrame con las dos columnas de precision@k indexado por pregunta.

In [ ]:
def chunk_secciones(texto):
    # TODO: separa por "\n## " y antepone el título del documento (primera línea, "# ...") a cada sección
    return [texto]

idx_secc = indexar(chunk_secciones)
df_secc = precision_at_k(idx_secc)
# TODO: compara df_fijo y df_secc pregunta por pregunta


### Ejercicio 2 – Una pregunta multi-hop que RAG no resuelve bien

Haz la pregunta: *"Si confirmo un lunes a las 16:00 un pedido de 3 paquetes de 3 kilos cada uno a Temuco, ¿cuánto pago de despacho en total y qué día de la semana llega?"*. Responder bien exige **dos documentos** (tarifas y despachos), **aritmética** (3 × tarifa) y **razonar sobre el corte de las 15:00**. Analiza con `verbose=True`: ¿recuperó los fragmentos necesarios? ¿Calculó bien? ¿Qué parte falla: recuperación, generación o ambas? Propón una mitigación (pista: en la clase 5 la recuperación será una tool de un agente).

**Pistas:** Mira primero qué fragmentos llegaron (`verbose=True`) y recién después juzga la respuesta. Un error de cálculo con los fragmentos correctos es un problema de generación; un fragmento faltante es un problema de recuperación.

In [ ]:
pregunta = "Si confirmo un lunes a las 16:00 un pedido de 3 paquetes de 3 kilos cada uno a Temuco, ¿cuánto pago de despacho en total y qué día de la semana llega?"
# TODO: usa responder_rag(idx_secc, pregunta, k=3, verbose=True) y analiza; prueba también con k=6


### Ejercicio 3 – Documentos contradictorios: la vigencia no está en el embedding

Pregunta *"¿Cuánto cuesta despachar un paquete de 3 kilos?"* y revisa qué fragmentos llegan. Luego indexa de nuevo agregando a la metadata `vigente: True/False` (falso para el documento cuyo nombre contiene `antigua`) con `indexar(chunk_secciones, meta_extra=...)`, y busca con `filtro=lambda m: m["vigente"]`. Compara los fragmentos recuperados y la respuesta en ambos casos, y vuelve a calcular precision@3 con el filtro.

**Pistas:** `meta_extra` recibe el nombre del documento y devuelve un dict que se agrega a la metadata de cada chunk. El filtro se aplica al recuperar, no al indexar.

In [ ]:
# TODO: idx_v2 = indexar(chunk_secciones, meta_extra=lambda doc: {"vigente": "antigua" not in doc})
#       compara idx_v2.buscar(pregunta) con y sin filtro=lambda m: m["vigente"], y precision_at_k(idx_v2, filtro=...)


### Ejercicio 4 – Abierto: ¿qué documentos de tu organización servirían para RAG?

Lista 3 fuentes de tu trabajo que serían buenas candidatas para RAG y 2 que serían malas, explicando por qué según las limitaciones vistas hoy (pérdida de contexto al trocear, preguntas de agregación o multi-hop, documentos obsoletos o contradictorios, tablas y PDFs escaneados, permisos de acceso). Si tienes un documento corto a mano (sin datos confidenciales), agrégalo a `DOCS`, reindexa y haz 3 preguntas.

In [ ]:
# Escribe tu respuesta en esta celda como comentario o en una celda de texto


## Para seguir en casa

- Tu key gratuita de Gemini sigue funcionando fuera de clase: los límites diarios se reinician cada día (medianoche hora del Pacífico). Puedes ver el uso y los límites de tu proyecto en [AI Studio](https://aistudio.google.com/).
- Si quieres probar otro modelo, cambia `MODEL` (por ejemplo a `"gemini-3.8-flash"`), pero recuerda que los modelos 3.6+ ignoran `temperature`.
- Recuerda la regla del nivel gratuito: nada de datos personales ni confidenciales en los prompts.
- **Lectura relacionada:** Alto (2025), capítulo sobre conocimiento/RAG; Brousseau & Sharp (2024), sección sobre RAG en producción.
